In [ ]:
import pickle
import h5py
import numpy as np
from sklearn.ensemble import VotingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

HSA

In [ ]:
#Load models
def load_model(model_name, protein_name):

    with open(f'./trained_models/{protein_name}/{protein_name}_{model_name}.pkl', 'rb') as f:
        model = pickle.load(f)
    return model

cnn_model = load_model('CNN', 'HSA')
chembert_model = load_model('ChemBert', 'HSA')
molformer_model = load_model('MolFormer', 'HSA')
lightgbm_model = load_model('lightgbm', 'HSA')


In [ ]:
#Load validation sets
def load_val(model_name, protein_name, features_name):
    with h5py.File('./intermediates/embeddings/{protein_name}_{model_name}_val.h5', 'r') as f:
        features = f[f'{features_name}'][:]
        labels = f['labels'][:]
    return features, labels

chembert_features, chembert_labels = load_val('ChemBert', 'HSA', 'embeddings')
molformer_features, molformer_labels = load_val('MolFormer', 'HSA', 'embeddings')
cnn_features, cnn_labels = load_val('MolFormer', 'HSA', 'encoded_smiles')
lightgbm_features, lightgbm_labels = load_val('lightgbm', 'HSA', 'fingerprints')

In [ ]:
assert np.array_equal(cnn_labels, chembert_labels)
assert np.array_equal(lightgbm_labels, molformer_labels)
assert np.array_equal(chembert_labels, lightgbm_labels)

In [ ]:
voting_classifier = VotingClassifier(
    estimators=[
        ('cnn', cnn_model),
        ('chembert', chembert_model),
        ('molformer', molformer_model),
        ('lightgbm', lightgbm_model)
    ],
    voting='soft'
)

In [ ]:
if voting_classifier.voting == 'soft':
    final_predictions = voting_classifier.predict(np.hstack([
        cnn_model.predict_proba(cnn_features),
        chembert_model.predict_proba(chembert_features),
        molformer_model.predict_proba(molformer_labels),
        lightgbm_model.predict_proba(lightgbm_features)
    ]))

In [ ]:
base_models = [
    ('cnn', cnn_model),
    ('chembert', chembert_model),
    ('molformer', molformer_model),
    ('lightgbm', lightgbm_model)
]


meta_model = LogisticRegression()

stacking_classifier = StackingClassifier(estimators=base_models, final_estimator=meta_model)

stacking_classifier.fit(val_features, val_labels)